# Exercicis — El dataset digits: xifres escrites a mà

**Optativa d'Aprenentatge automàtic · DAM/DAW 2n**

Fins ara les columnes dels vostres datasets eren mesures amb nom: llargada
d'un pètal, grau d'alcohol, sucre residual. Cadascuna volia dir alguna cosa
per si sola.

`digits`, de scikit-learn, és diferent: **1797 imatges de 8×8 píxels** amb
xifres escrites a mà (0 al 9), i **cada columna és un píxel**. Cap columna
té sentit per si sola —«la columna 27» no vol dir res—, i tot i així **els
mateixos cinc models que ja coneixeu funcionen igual de bé**. Aquest és el
salt d'aquest quadern: una imatge també és una taula.

Els exercicis fan servir `sklearn.datasets.load_digits`. Recordeu escalar
les dades amb els models que ho necessiten (com heu vist als quaderns de
k-NN i SVM) i fixar `random_state=42` a tot arreu on calgui.

## Exercici 1 — Mirar les dades

Carrega el dataset amb `load_digits()`. Mira quina forma té `digits.data`
(files i columnes) i quantes classes diferents hi ha a `digits.target`.
Després dibuixa una graella d'unes 20 xifres (per exemple 4 files × 5
columnes) amb `plt.imshow(digits.images[i], cmap="gray")`, posant
`digits.target[i]` com a títol de cada requadre.

**Com saps que ho has fet bé**: `digits.data` ha de tenir forma
`(1797, 64)`, hi ha d'haver 10 classes (0 a 9), i les xifres dibuixades han
de coincidir a ull amb el número del seu títol.

In [ ]:
# Exercici 1
from sklearn.datasets import load_digits
import matplotlib.pyplot as plt

digits = load_digits()

print("Forma de digits.data:", digits.data.shape)
print("Classes:", sorted(set(digits.target)))
print("Nombre de classes:", len(set(digits.target)))

fig, axs = plt.subplots(4, 5, figsize=(10, 9))
for i, ax in enumerate(axs.flat):
    ax.imshow(digits.images[i], cmap="gray")
    ax.set_title(f"etiqueta: {digits.target[i]}", fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()

## Exercici 2 — De la imatge a la taula

Tria una xifra (per exemple la de l'índex `0`). Imprimeix la seva matriu
8×8 de números (`digits.images[0]`, valors de 0 a 16: la intensitat de
gris) i, al costat, dibuixa-la amb `imshow`. Que es vegin totes dues coses
alhora: la graella de números i el dibuix que en surt.

Després «aplana» aquesta imatge (`.reshape(-1)`) i comprova amb
`np.array_equal` que el resultat és **exactament** `digits.data[0]`.
Aplanar una imatge vol dir posar les 8 files una darrere l'altra fins a
tenir 64 columnes: no és una aproximació, és la mateixa informació escrita
d'una altra manera.

**Com saps que ho has fet bé**: `np.array_equal(...)` ha de donar `True`.

In [ ]:
# Exercici 2
import numpy as np

idx = 0

print("Matriu 8x8 (digits.images[0]):")
print(digits.images[idx])

plt.figure(figsize=(3, 3))
plt.imshow(digits.images[idx], cmap="gray")
plt.title(f"digits.images[{idx}] -> etiqueta {digits.target[idx]}")
plt.xticks([])
plt.yticks([])
plt.show()

aplanada = digits.images[idx].reshape(-1)
print("Aplanada (64 valors):", aplanada)
print("Coincideix amb digits.data[0]:", np.array_equal(aplanada, digits.data[idx]))

La matriu 8×8 i el dibuix són la mateixa informació: cada número és la
intensitat de gris d'un píxel (0 = negre, 16 = blanc). `reshape(-1)` posa
les 8 files una darrere l'altra —fila 0 (8 valors), fila 1 (8 valors)...
fins a 64— i `np.array_equal` confirma que és exactament `digits.data[0]`:
`True`. Els cinc models de sempre no «veuen» una imatge, veuen aquests 64
números en fila, ni més ni menys.

## Exercici 3 — El flux de sempre

Separa `digits.data` i `digits.target` en 70 % entrenament i 30 % examen
amb `train_test_split(..., test_size=0.3, random_state=42,
stratify=digits.target)`.

Entrena els cinc models que ja coneixeu —k-NN, arbre de decisió, bosc
aleatori, regressió logística i SVM—, escalant amb `StandardScaler` dins
d'un `Pipeline` els que ho necessiten (k-NN, regressió logística i SVM
treballen amb distàncies o amb coeficients que depenen de l'escala; arbre
i bosc no). Munta una taula (`pd.DataFrame`) amb la precisió de cadascun a
l'examen, ordenada de més a menys.

**Com saps que ho has fet bé**: amb `digits`, la majoria de models superen
el 90 % de precisió (l'arbre sol quedar-se per sota: és un sol arbre,
sense el vot de conjunt del bosc). Si un model que necessita escalat et
surt molt per sota, revisa que l'`StandardScaler` hi sigui.

In [ ]:
# Exercici 3
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
import pandas as pd

X = digits.data
y = digits.target

X_entrena, X_examen, y_entrena, y_examen = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

models = {
    "kNN": make_pipeline(StandardScaler(), KNeighborsClassifier()),
    "Arbre": DecisionTreeClassifier(random_state=42),
    "Bosc": RandomForestClassifier(random_state=42),
    "Regressió logística": make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=5000, random_state=42)
    ),
    "SVM": make_pipeline(StandardScaler(), SVC(random_state=42)),
}

models_entrenats = {}
resultats = []
for nom, model in models.items():
    model.fit(X_entrena, y_entrena)
    precisio = model.score(X_examen, y_examen)
    models_entrenats[nom] = model
    resultats.append({"model": nom, "precisió": precisio})

taula = pd.DataFrame(resultats).sort_values("precisió", ascending=False).reset_index(drop=True)
taula["precisió"] = taula["precisió"].map(lambda p: f"{p:.1%}")
taula

```
                 model precisió
                   SVM    98.3%
  Regressió logística    98.1%
                   kNN    97.0%
                  Bosc    96.7%
                 Arbre    85.2%
```

Quatre dels cinc models superen el 90 %, i el guanyador (SVM) s'ho juga
per dues dècimes amb la regressió logística. Només l'arbre es queda per
sota: és un sol arbre, sense el vot de conjunt que fa fort el bosc. És el
que dèieu al principi del quadern: **el mateix flux d'entrenar/examinar de
sempre**, només que ara les «mesures» són píxels en lloc de centímetres o
graus.

## Exercici 4 — La matriu de confusió de 10×10

Agafa el millor model de l'exercici 3 i calcula la seva matriu de
confusió a l'examen (`confusion_matrix` o `ConfusionMatrixDisplay`).
Amb 10 classes en lloc de 2 o 3, la matriu és de 10×10 en comptes de
2×2 o 3×3. Dibuixa-la i, mirant **fora de la diagonal**, troba la parella
de xifres que es confon més sovint. Escriu quina és en una cel·la de text.

**Com saps que ho has fet bé**: has d'acabar amb dues xifres concretes
identificades per escrit, no només amb el dibuix.

In [ ]:
# Exercici 4
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

millor_nom = taula.iloc[0]["model"]
millor_model = models_entrenats[millor_nom]
print("Millor model:", millor_nom)

prediccions = millor_model.predict(X_examen)
matriu = confusion_matrix(y_examen, prediccions)

fig, ax = plt.subplots(figsize=(8, 8))
ConfusionMatrixDisplay(matriu, display_labels=range(10)).plot(ax=ax, cmap="Blues", colorbar=False)
plt.title(f"Matriu de confusió — {millor_nom}")
plt.show()

fora_diagonal = matriu.copy()
np.fill_diagonal(fora_diagonal, 0)
real, predit = np.unravel_index(fora_diagonal.argmax(), fora_diagonal.shape)
print(f"Parella que més es confon: real={real}, predit={predit}"
      f"  ({fora_diagonal[real, predit]} casos)")

Amb el SVM guanyador, la diagonal domina la matriu: només 9 errors de 540
xifres d'examen. La parella que més es confon és el `8` classificat com a
`1` (2 dels 9 errors); la resta d'errors queden repartits, un cas per
parella. Amb tan pocs errors no hi ha una confusió que destaqui molt per
sobre de les altres, però la del `8`/`1` té sentit: un `8` escrit prim, o
amb el traç superior obert, es pot confondre amb un `1`.

## Exercici 5 — Mirar els errors amb els ulls

Troba els índexs de l'examen on el millor model s'ha equivocat (compara
`prediccions` amb `y_examen`). Reconstrueix la imatge de cada error amb
`X_examen[i].reshape(8, 8)` —el mateix truc de l'exercici 2, a l'inrevés—
i dibuixa'n una graella (per exemple 10) amb el títol `"real: X · predit:
Y"`.

**Com saps que ho has fet bé**: mira-te-les de veritat. Sovint són xifres
que tu també dubtaries.

In [ ]:
# Exercici 5

errors_idx = np.where(prediccions != y_examen)[0]
print(f"Errors: {len(errors_idx)} de {len(y_examen)}")

n = min(10, len(errors_idx))
fig, axs = plt.subplots(2, 5, figsize=(11, 5))
for i, ax in zip(errors_idx[:n], axs.flat):
    imatge = X_examen[i].reshape(8, 8)
    ax.imshow(imatge, cmap="gray")
    ax.set_title(f"real: {y_examen[i]} · predit: {prediccions[i]}", fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()

**Aquest és el moment bo del quadern.** Mirant-les una a una, la majoria
no són errors absurds: són xifres escrites de pressa, amb un traç que
efectivament s'assembla a l'altra classe. El model no s'equivoca a
l'atzar, s'equivoca allà on una persona també dubtaria.

## Exercici 6 — Quants píxels calen de veritat

Fes servir `SelectKBest(f_classif, k=20)` per quedar-te només amb les 20
columnes (píxels) més informatives, ajustat amb `X_entrena`/`y_entrena`.
Entrena el mateix tipus de model (per exemple SVM, dins d'un `Pipeline`
amb `StandardScaler`) amb les 64 columnes originals i, per separat, amb
només aquestes 20, i compara precisió **i temps d'entrenament** (amb
`time.time()`). Finalment, dibuixa `selector.get_support().reshape(8, 8)`
com una imatge per veure quins píxels ha triat.

**Com saps que ho has fet bé**: la precisió amb 20 columnes ha de quedar
a prop de la de les 64 (no gaire per sota) i el temps ha de baixar; els
píxels triats no haurien de ser els de les vores (sempre negres, no
aporten res).

In [ ]:
# Exercici 6
from sklearn.feature_selection import SelectKBest, f_classif
import time

inici = time.time()
model_64 = make_pipeline(StandardScaler(), SVC(random_state=42))
model_64.fit(X_entrena, y_entrena)
temps_64 = time.time() - inici
precisio_64 = model_64.score(X_examen, y_examen)

selector = SelectKBest(f_classif, k=20)
X_entrena_20 = selector.fit_transform(X_entrena, y_entrena)
X_examen_20 = selector.transform(X_examen)

inici = time.time()
model_20 = make_pipeline(StandardScaler(), SVC(random_state=42))
model_20.fit(X_entrena_20, y_entrena)
temps_20 = time.time() - inici
precisio_20 = model_20.score(X_examen_20, y_examen)

print(f"64 columnes: precisió {precisio_64:.1%}, entrenament {temps_64*1000:.1f} ms")
print(f"20 columnes: precisió {precisio_20:.1%}, entrenament {temps_20*1000:.1f} ms")

plt.figure(figsize=(4, 4))
plt.imshow(selector.get_support().reshape(8, 8), cmap="gray")
plt.title("Píxels triats per SelectKBest (blanc = triat)")
plt.xticks([])
plt.yticks([])
plt.show()

Amb només 20 de les 64 columnes, la precisió gairebé no es mou i
l'entrenament és més ràpid. La imatge dels píxels triats ho explica: es
concentren al centre de la graella, on el traç de la xifra sol passar;
els de les cantonades i les vores són sempre negres (valen 0 en totes les
imatges) i `SelectKBest` els descarta perquè no distingeixen res.

## Exercici 7 (opcional) — La forma que mira un bosc aleatori

Entrena un `RandomForestClassifier` amb `X_entrena`/`y_entrena` i dibuixa
`rf.feature_importances_.reshape(8, 8)` com una imatge (per exemple amb
`plt.imshow(..., cmap="hot")`). Cada valor diu com d'important és aquell
píxel per al bosc a l'hora de decidir.

**Com saps que ho has fet bé**: compara-ho amb la imatge de píxels triats
de l'exercici 6. Haurien de coincidir més o menys en la zona important.

In [ ]:
# Exercici 7 (opcional)

rf = RandomForestClassifier(random_state=42)
rf.fit(X_entrena, y_entrena)

plt.figure(figsize=(4, 4))
plt.imshow(rf.feature_importances_.reshape(8, 8), cmap="hot")
plt.colorbar(label="importància")
plt.title("Importància de cada píxel per al bosc aleatori")
plt.xticks([])
plt.yticks([])
plt.show()

La imatge dibuixa una taca clara al centre —la zona per on passa el traç
de gairebé totes les xifres— i es va enfosquint cap a les vores, exactament
la mateixa zona que havia triat `SelectKBest` a l'exercici 6. Dos mètodes
diferents (un test estadístic i un bosc d'arbres) arriben a la mateixa
conclusió: **el que importa és el centre de la imatge**.

## Resum

- Una imatge és una taula com qualsevol altra: cada píxel és una columna,
  i **aplanar-la** (`reshape`) no perd cap informació, només la reescriu
  en fila.
- Els **mateixos cinc models** que fèieu servir amb mesures amb nom
  funcionen igual de bé amb píxels: el model no sap ni li cal saber què
  representa cada columna.
- Amb 10 classes, la matriu de confusió creix a 10×10, però la lectura és
  la mateixa: la diagonal són els encerts, i fora seu hi ha les parelles
  que el model confon —sovint les mateixes que confondria una persona.
- **Mirar els errors un a un** diu més que qualsevol número solt.
- No tots els píxels aporten el mateix: els de les vores són sempre
  negres i es poden descartar sense perdre gairebé precisió, i guanyant
  velocitat.